
# YOLOv8 Binary Object Detection and Cropping Pipeline

This notebook includes:

1. Converting YOLO dataset annotations from multi-class to binary.
2. Fine-tuning YOLOv8 for binary detection.
3. Inference pipeline (detection -> cropping -> augmentation -> classification preparation).

Abdulmlik almuhanna

In [2]:
!pip install -q ultralytics roboflow sahi keras

!pip install -q glob2
import os
from sahi.utils.file import download_from_url
# The following import is causing a ModuleNotFoundError and is not used later in the notebook.
# from sahi.utils.ultralytics import download_yolo11n_model
from ultralytics import YOLO
from roboflow import Roboflow
import cv2
import numpy as np
from glob import glob

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:

#from roboflow import Roboflow
#rf = Roboflow(api_key="TS1niacLXvvWTierCKCT")
#project = rf.workspace("insectai").project("insects_noclasses")
#version = project.version(7)
#dataset = version.download("yolov8")
'''
rf = Roboflow(api_key="TS1niacLXvvWTierCKCT")
project = rf.workspace("insectai").project("segtoboxes_noclass")
version = project.version(3)
dataset = version.download("yolov8")





# the SF sliced dataset:
# this is done after running the Sahi slicing script below but i uploaded the sliced images to Roboflow
# to make it easier to download the dataset and change the annotations to YOLO format
rf = Roboflow(api_key="TS1niacLXvvWTierCKCT")
project = rf.workspace("insectai").project("experiment-pzbwo")
version = project.version(4)
dataset = version.download("yolov8")'''

!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="TS1niacLXvvWTierCKCT")
project = rf.workspace("insectai").project("new_images_single_annotation")
version = project.version(2)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to New_images_single_annotation-2 in yolov8:: 100%|██████████| 438/438 [00:00<00:00, 3550.80it/s]


Sahi slicers

### training the binary detector

In [14]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

# Download the YOLO model from Hugging Face to the /content/ directory
model_path = hf_hub_download("davsolai/yolo11x-p2-coco", "model.pt", local_dir="/content/")
print(f"Model downloaded to: {model_path}")

Model downloaded to: /content/model.pt


In [12]:
import yaml

# Define hyperparameters for the detector model
hyperparams = { # Replace this dynamically if needed
    'epochs': 200,
    'imgsz': 1504,
    'batch': 2,
    'project': 'binary_detector',
    'max_det': 600,
    'single_cls':True,
    'visualize':True,
    'save_crop':True,
    'show_labels':False,
    'warmup_epochs':20.0,
}

# Save the hyperparameters to a YAML file
with open("cgs.yaml", "w") as file:
  yaml.dump(hyperparams, file)
print("Hyperparameters saved to 'cgs.yaml'.")

Hyperparameters saved to 'cgs.yaml'.


In [ ]:
# Define model path and load the model
model = YOLO(model_path)

In [ ]:
# Fine-tune YOLO on binary dataset
model.train(data="/content/New_images_single_annotation-2/data.yaml",cfg="/content/cgs.yaml")

Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=/content/cgs.yaml, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/New_images_single_annotation-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1504, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=600, mixup=0.0, mode=train, model=/content/model.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overla

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d0342d788f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
#get validation results and save the model
model.val(save=True)

Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
YOLO11x-p2 summary (fused): 235 layers, 57,739,092 parameters, 0 gradients, 242.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2581.8±977.5 MB/s, size: 243.5 KB)
val: Scanning /content/New_images_single_annotation-2/valid/labels.cache... 5 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5/5 2.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.6it/s 0.6s
                   all          5        690      0.803      0.755      0.796      0.386
Speed: 4.1ms preprocess, 48.4ms inference, 0.0ms loss, 2.8ms postprocess per image
Results saved to /content/runs/detect/val


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d02b363d460>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 